# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Asiya-Akhtar/flyrank-ml/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

I chose correlation and signal analysis rather than a supervised classification model because this dataset does not contain a separate prediction label. My Week-4 baseline was a rule-based prioritization system built from current-state signals, while `trend_direction` was used only as an outcome for signal auditing. Signal analysis therefore fits the available data without inventing a target or using an outcome-derived field as a model feature.

The goal is to measure whether the current-state signals used by the baseline have a meaningful relationship with the observed outcome and whether the signals provide enough evidence to justify the baseline's prioritization logic. I will treat the results as directional decision-support evidence rather than causal claims.

In [9]:
# Load the dataset from GitHub
import pandas as pd
import numpy as np

DATA_URL = "https://raw.githubusercontent.com/Asiya-Akhtar/flyrank-ml/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_URL)

print("Rows:", len(df))
print("Columns:", len(df.columns))

# Confirm the fields used by the Week-4 baseline and its audit outcome
required_columns = [
    "content_id",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "trend_direction",
    "trend_pct"
]

missing = [c for c in required_columns if c not in df.columns]

print("\nMissing required columns:", missing)

assert not missing, f"Missing columns: {missing}"

print("\nData check: PASSED")

Rows: 30000
Columns: 44

Missing required columns: []

Data check: PASSED


In [10]:
# Recreate the exact Week-4 baseline signals

df["stale"] = (
    df["days_since_last_update"] >= 90
)

df["low_ctr_visible"] = (
    (df["impressions_90d"] >= 500)
    & (df["avg_position"].between(1, 20))
    & (df["ctr"] < 0.5)
)

df["baseline_score"] = (
    2 * df["stale"].astype(int)
    + 1 * df["low_ctr_visible"].astype(int)
)

print("Baseline score distribution:")
print(df["baseline_score"].value_counts().sort_index())

print("\nSignal counts:")
print("Stale:", df["stale"].sum())
print("Low CTR visible:", df["low_ctr_visible"].sum())

Baseline score distribution:
baseline_score
0    14439
1     6216
2     5816
3     3529
Name: count, dtype: int64

Signal counts:
Stale: 9345
Low CTR visible: 9745


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

This analysis does not train a supervised prediction model because the dataset does not provide a separate prediction label. I therefore use the full 30,000-row snapshot for descriptive signal analysis, matching the population used by my Week-4 baseline. I do not use future-window fields as inputs. The observed `trend_direction` field is treated only as the audited outcome, consistent with Week 4.

In [13]:
# Confirm the analysis population and check for missing values

analysis_columns = [
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "trend_direction",
    "trend_pct"
]

print("Analysis rows:", len(df))

print("\nMissing values:")
print(df[analysis_columns].isna().sum())

print("\nTrend direction distribution:")
print(df["trend_direction"].value_counts(dropna=False))

Analysis rows: 30000

Missing values:
days_since_last_update       0
impressions_90d              0
avg_position                 0
ctr                          0
trend_direction              0
trend_pct                 3388
dtype: int64

Trend direction distribution:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


### Signal analysis

I compare the observed declining rate across the baseline's signal buckets. This tests whether the signals that drove the Week-4 rule continue to show directional separation in the same dataset. A difference between groups is evidence of association, not proof that the signal causes decline.

In [14]:
# Signal 1: staleness

staleness_analysis = (
    df.groupby("stale")
      .agg(
          n=("content_id", "size"),
          declining_rate=("trend_direction",
                           lambda x: (x == "down").mean() * 100)
      )
      .reset_index()
)

staleness_analysis["bucket"] = staleness_analysis["stale"].map({
    False: "Not stale (<90 days)",
    True: "Stale (90+ days)"
})

staleness_analysis[
    ["bucket", "n", "declining_rate"]
]

,bucket,n,declining_rate
0,Not stale (<90 days),20655,51.203099
1,Stale (90+ days),9345,60.845372


In [15]:
# Signal 2: low CTR despite visibility

ctr_analysis = (
    df.groupby("low_ctr_visible")
      .agg(
          n=("content_id", "size"),
          declining_rate=("trend_direction",
                           lambda x: (x == "down").mean() * 100)
      )
      .reset_index()
)

ctr_analysis["bucket"] = ctr_analysis["low_ctr_visible"].map({
    False: "Not low-CTR visible",
    True: "Low-CTR visible"
})

ctr_analysis[
    ["bucket", "n", "declining_rate"]
]

,bucket,n,declining_rate
0,Not low-CTR visible,20255,50.140706
1,Low-CTR visible,9745,62.657773


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

### Model training

I use Logistic Regression to estimate the probability that an item is observed as declining. The model uses only current-state fields that were available to the Week-4 baseline: days since last update, impressions over 90 days, average position, and CTR. `trend_direction` is used only as the outcome for training and evaluation and is never included as a feature.

I use a stratified train/test split so both classes are represented in the evaluation set. The Week-4 rule is evaluated on the exact same held-out rows. I compare the two approaches using Precision@10 and Precision@50, matching the ranking-oriented evaluation used for the baseline.

In [17]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Current-state features only
features = [
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr"
]

# Outcome: observed downward trend
model_df = df[features + ["trend_direction"]].dropna().copy()

X = model_df[features]
y = (model_df["trend_direction"] == "down").astype(int)

# Honest held-out evaluation split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("Training decline rate:", round(y_train.mean() * 100, 2), "%")
print("Test decline rate:", round(y_test.mean() * 100, 2), "%")

Training rows: 24000
Test rows: 6000
Training decline rate: 54.21 %
Test decline rate: 54.2 %


In [18]:
# Train Logistic Regression

model = Pipeline([
    ("scale", StandardScaler()),
    ("logistic", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

model.fit(X_train, y_train)

# Probability of observed decline
model_probability = model.predict_proba(X_test)[:, 1]

print("Model trained successfully.")
print("Predictions generated:", len(model_probability))

Model trained successfully.
Predictions generated: 6000


### STEP 9 — Calculate Precision@10 and Precision@50

In [19]:
def precision_at_k(y_true, scores, k):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)

    order = np.argsort(-scores)
    top_k = y_true[order[:k]]

    return top_k.mean()

# Model ranking
model_p10 = precision_at_k(y_test.values, model_probability, 10)
model_p50 = precision_at_k(y_test.values, model_probability, 50)

print("Logistic Regression Precision@10:",
      round(model_p10 * 100, 2), "%")

print("Logistic Regression Precision@50:",
      round(model_p50 * 100, 2), "%")

Logistic Regression Precision@10: 40.0 %
Logistic Regression Precision@50: 46.0 %


In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Recreate the Week-4 baseline score on the held-out test rows

baseline_test = X_test.copy()

baseline_test["baseline_score"] = (
    2 * (
        baseline_test["days_since_last_update"] >= 90
    ).astype(int)
    +
    (
        (baseline_test["impressions_90d"] >= 500)
        &
        (baseline_test["avg_position"].between(1, 20))
        &
        (baseline_test["ctr"] < 0.5)
    ).astype(int)
)

baseline_p10 = precision_at_k(
    y_test.values,
    baseline_test["baseline_score"].values,
    10
)

baseline_p50 = precision_at_k(
    y_test.values,
    baseline_test["baseline_score"].values,
    50
)

print("Week-4 baseline Precision@10:",
      round(baseline_p10 * 100, 2), "%")

print("Week-4 baseline Precision@50:",
      round(baseline_p50 * 100, 2), "%")

Week-4 baseline Precision@10: 70.0 %
Week-4 baseline Precision@50: 54.0 %


In [21]:
comparison = pd.DataFrame({
    "method": [
        "Week-4 baseline",
        "Logistic Regression"
    ],
    "precision_at_10": [
        baseline_p10,
        model_p10
    ],
    "precision_at_50": [
        baseline_p50,
        model_p50
    ]
})

comparison["precision_at_10"] = (
    comparison["precision_at_10"] * 100
).round(2)

comparison["precision_at_50"] = (
    comparison["precision_at_50"] * 100
).round(2)

comparison

,method,precision_at_10,precision_at_50
0,Week-4 baseline,70.0,54.0
1,Logistic Regression,40.0,46.0


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

The model and baseline are evaluated on the same held-out rows. I treat the comparison as measured decision-support evidence rather than proof of causation.

I inspect the highest-scored model cases and compare them with the observed outcome. I also inspect false-positive and false-negative cases to understand where the model is uncertain or wrong.

The model's feature coefficients are interpreted as associations with the model's prediction, not as causal effects. A weaker result than the baseline would be useful evidence that the added model complexity does not improve this lane.

In [22]:
# Build an error-analysis table

error_df = X_test.copy()

error_df["actual_decline"] = y_test.values
error_df["model_probability"] = model_probability
error_df["model_prediction"] = (
    model_probability >= 0.5
).astype(int)

error_df["error_type"] = np.select(
    [
        (error_df["actual_decline"] == 1) &
        (error_df["model_prediction"] == 0),

        (error_df["actual_decline"] == 0) &
        (error_df["model_prediction"] == 1)
    ],
    [
        "false_negative",
        "false_positive"
    ],
    default="correct"
)

print(error_df["error_type"].value_counts())

display(
    error_df[
        error_df["error_type"] != "correct"
    ].head(10)
)

error_type
correct           3339
false_positive    2251
false_negative     410
Name: count, dtype: int64


,days_since_last_update,impressions_90d,avg_position,ctr,actual_decline,model_probability,model_prediction,error_type
8732,20,1,0.0,0.00,0,0.551264,1,false_positive
2110,104,511,15.0,0.39,0,0.612723,1,false_positive
9902,20,127,18.3,0.00,0,0.522877,1,false_positive
24640,20,1,0.0,0.00,0,0.551264,1,false_positive
25967,20,373,60.4,0.00,1,0.457363,0,false_negative
26973,20,595,5.2,0.17,0,0.540117,1,false_positive
28981,26,16902,10.7,0.42,0,0.516049,1,false_positive
24765,104,40445,15.7,0.17,0,0.570858,1,false_positive
7188,20,845,37.1,0.00,1,0.492892,0,false_negative
11694,22,11,34.9,0.00,1,0.499467,0,false_negative


In [23]:
# Inspect Logistic Regression coefficients

coefficients = pd.DataFrame({
    "feature": features,
    "coefficient": model.named_steps["logistic"].coef_[0]
})

coefficients["absolute_coefficient"] = (
    coefficients["coefficient"].abs()
)

coefficients = coefficients.sort_values(
    "absolute_coefficient",
    ascending=False
)

coefficients[
    ["feature", "coefficient"]
]

,feature,coefficient
3,ctr,-0.188152
0,days_since_last_update,0.185994
2,avg_position,-0.093853
1,impressions_90d,-0.075830


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.